# 02 - Skapa och validera target-variabeln

## Syfte

Syftet med denna notebook är att skapa target-variabeln `late`, som anger om en order levererades efter det beräknade leveransdatumet.

ML-tabellen från föregående notebook läses in som artifact. Därefter kontrolleras de datumkolumner som behövs för labeln, target-variabeln skapas och valideras på riktiga orders.

Fördelningen mellan sena och icke-sena orders undersöks för att bedöma om datasetet har class imbalance.

Slutligen sparas endast orders med en känd target som ett nytt artifact för nästa notebook.

In [40]:
from pathlib import Path
import pandas as pd
import numpy as np

In [ ]:


# Sökväg till artifacts från föregående notebook
artifacts_dir = Path("artifacts")

# Läs in ML-tabellen från Notebook 01
ml_table = pd.read_csv(
    artifacts_dir / "ml_table.csv"
)

print("Shape:", ml_table.shape)
ml_table.head()

Shape: (99441, 20)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,total_items,total_price,total_freight,total_payment_value,number_of_payments,max_installments,customer_zip_code_prefix,customer_state,seller_count,seller_zip_code_prefix,seller_state,distance_km
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1.0,29.99,8.72,38.71,3.0,1.0,3149,SP,1.0,9350.0,SP,18.576110
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,1.0,118.70,22.76,141.46,1.0,1.0,47813,BA,1.0,31570.0,SP,851.495069
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,1.0,159.90,19.22,179.12,1.0,3.0,75265,GO,1.0,14840.0,SP,514.410666
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,1.0,45.00,27.20,72.20,1.0,1.0,59296,RN,1.0,31842.0,MG,1822.226335
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,1.0,19.90,8.72,28.62,1.0,1.0,9195,SP,1.0,8752.0,SP,29.676624


In [37]:
# Konvertera datumkolumner som används för target-variabeln
date_columns = [
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for column in date_columns:
    ml_table[column] = pd.to_datetime(
        ml_table[column],
        errors="coerce"
    )

print("Missing actual delivery date:",
      ml_table["order_delivered_customer_date"].isna().sum())

print("Missing estimated delivery date:",
      ml_table["order_estimated_delivery_date"].isna().sum())

Missing actual delivery date: 2965
Missing estimated delivery date: 0


## Kontroll av saknade leveransdatum

För att skapa target-variabeln `late` behöver vi både det faktiska
leveransdatumet och det uppskattade leveransdatumet.

Om det faktiska leveransdatumet saknas kan vi inte avgöra om ordern
levererades sent. Därför undersöker vi först vilka orderstatus som
har saknat faktiskt leveransdatum.

In [38]:
# Identifiera orders där faktiskt leveransdatum saknas
missing_delivery_status = (
    ml_table.loc[
        ml_table["order_delivered_customer_date"].isna(),
        "order_status"
    ]
    .value_counts()
)

# Visa hur många orders som saknar faktiskt leveransdatum
print(
    "Orders without actual delivery date:",
    ml_table["order_delivered_customer_date"].isna().sum()
)

# Kontrollera vilka orderstatus dessa orders har
print("\nOrder status:")
print(missing_delivery_status)

Orders without actual delivery date: 2965

Order status:
order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64


### Resultat: saknade faktiska leveransdatum

2965 orders saknar faktiskt leveransdatum.

De flesta av dessa orders har ännu inte en slutlig leveransstatus, exempelvis `shipped`, `processing`, `invoiced`, `canceled` eller `unavailable`.

Det finns även 8 orders med status `delivered` där faktiskt leveransdatum saknas. Dessa orders kan inte klassificeras som sena eller inte sena eftersom det faktiska leveransdatumet saknas.

Därför skapar vi endast target-variabeln för orders där både faktiskt och uppskattat leveransdatum finns.

In [41]:
# Skapa target-variabeln `late`
# 1 = leveransen skedde efter det uppskattade leveransdatumet
# 0 = leveransen skedde på eller före det uppskattade leveransdatumet

ml_table["late"] = np.where(
    ml_table["order_delivered_customer_date"].notna(),
    (
        ml_table["order_delivered_customer_date"]
        > ml_table["order_estimated_delivery_date"]
    ).astype(int),
    np.nan
)

print("Target created.")
print("\nMissing target:", ml_table["late"].isna().sum())
print("\nTarget distribution:")
print(ml_table["late"].value_counts(dropna=False))

Target created.

Missing target: 2965

Target distribution:
late
0.0    88649
1.0     7827
NaN     2965
Name: count, dtype: int64


## Kontroll av target på riktiga orders

Innan target-variabeln används vidare kontrollerar vi några konkreta
orders för att säkerställa att definitionen av `late` fungerar som tänkt.

En order ska få `late = 1` när det faktiska leveransdatumet är efter
det uppskattade leveransdatumet.

En order ska få `late = 0` när leveransen skedde på eller före det
uppskattade leveransdatumet.

In [42]:
# Visa ett urval av orders med båda leveransdatumen och den skapade targeten
label_check = (
    ml_table.loc[
        ml_table["late"].notna(),
        [
            "order_id",
            "order_status",
            "order_delivered_customer_date",
            "order_estimated_delivery_date",
            "late"
        ]
    ]
    .sample(10, random_state=42)
)

label_check

,order_id,order_status,order_delivered_customer_date,order_estimated_delivery_date,late
22500,c58cff333993bb6b7161d7ec1350eef3,delivered,2018-04-06 02:32:49,2018-04-18,0.0
68941,87673b5ccb20de0a91c28cc461105d76,delivered,2018-05-23 15:28:28,2018-05-30,0.0
23988,80b430d0029bb33110ac31d60e87e0b8,delivered,2017-12-07 18:43:46,2017-12-27,0.0
31303,580603672a21252f21fa8a8b4ca85986,delivered,2018-04-26 17:44:27,2018-05-08,0.0
36131,c09f32e7ba9b4a134455b36eeff8fff3,delivered,2018-04-13 17:32:07,2018-04-24,0.0
58330,462240cb1ec4e5db517e73fff57ebfc0,delivered,2017-12-14 18:56:14,2017-12-20,0.0
95960,5bc2a7b8f0817443a86b69461e742cc9,delivered,2018-01-12 21:59:18,2018-02-01,0.0
56131,4ef8f514f95bb0a41f58965ea04e7027,delivered,2017-05-26 15:59:47,2017-06-07,0.0
56750,587904dc1c873ebfb6078160b4819d8e,delivered,2017-12-06 19:43:34,2017-12-15,0.0
92212,29c3b79aace1b72a82b1232bf494e16f,delivered,2018-04-28 15:51:50,2018-01-24,1.0


## Verifiering av target-variabeln

Förutom den manuella kontrollen verifierar vi target-variabeln programmatiskt.

Kontrollen säkerställer att:

- `late = 1` endast när faktisk leverans skedde efter beräknat leveransdatum.
- `late = 0` när faktisk leverans skedde på eller före beräknat leveransdatum.
- orders utan faktiskt leveransdatum har saknad target.

In [43]:
# Kontrollera att target-variabeln följer den definierade regeln
valid_label_rows = ml_table["late"].notna()

expected_late = (
    ml_table.loc[valid_label_rows, "order_delivered_customer_date"]
    > ml_table.loc[valid_label_rows, "order_estimated_delivery_date"]
).astype(int)

label_is_correct = (
    ml_table.loc[valid_label_rows, "late"].astype(int)
    == expected_late
)

print("Correct labels:", label_is_correct.all())
print("Incorrect labels:", (~label_is_correct).sum())

Correct labels: True
Incorrect labels: 0


## Fördelning av target-klasser

Vi undersöker nu fördelningen mellan sena och icke-sena orders.

Detta behövs för att se om target-klasserna är balanserade eller om
datasetet har class imbalance. Resultatet används senare när vi väljer
lämpliga metrics och modeller.

In [44]:
# Räkna endast orders där target är känd
labeled_data = ml_table.loc[
    ml_table["late"].notna()
].copy()

# Räkna antal orders per target-klass
class_counts = labeled_data["late"].value_counts().sort_index()

# Beräkna procentandel per klass
class_percent = (
    labeled_data["late"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
)

print("Number of labeled orders:", len(labeled_data))

print("\nClass distribution:")
print(class_counts)

print("\nClass distribution (%):")
print(class_percent.round(2))

Number of labeled orders: 96476

Class distribution:
late
0.0    88649
1.0     7827
Name: count, dtype: int64

Class distribution (%):
late
0.0    91.89
1.0     8.11
Name: proportion, dtype: float64


### Bedömning av class imbalance

Target-klasserna är tydligt obalanserade.

91,89 % av de orders där target är känd har `late = 0`, medan endast
8,11 % har `late = 1`. Minoritetsklassen är därför betydligt mindre än
majoritetsklassen.

Detta innebär att accuracy ensam inte kommer att vara ett tillräckligt
bra mått för modellens prestanda. I senare steg behöver vi därför
använda metrics som exempelvis precision, recall och F1-score, med
särskilt fokus på klassen `late = 1`.

Ingen balansering av klasserna görs i denna notebook. Det hanteras
senare i samband med modellträning och utvärdering.

## Skapa dataset med känd target

För nästa steg behöver vi ett dataset där varje rad har en känd target.

Orders där faktiskt leveransdatum saknas kan inte klassificeras som
sena eller icke-sena och tas därför bort från det labelerade datasetet.

Den ursprungliga `ml_table` lämnas oförändrad i minnet, medan en separat
kopia med kända labels skapas och sparas som artifact.

In [45]:
# Skapa en separat kopia med endast orders där target är känd
labeled_ml_table = ml_table.loc[
    ml_table["late"].notna()
].copy()

# Target behöver vara heltalsvärden 0 och 1
labeled_ml_table["late"] = (
    labeled_ml_table["late"]
    .astype(int)
)

print("Labeled dataset shape:", labeled_ml_table.shape)
print("Unique order_id:", labeled_ml_table["order_id"].nunique())
print("Duplicate order_id:",
      labeled_ml_table["order_id"].duplicated().sum())

print("\nMissing target:",
      labeled_ml_table["late"].isna().sum())

Labeled dataset shape: (96476, 21)
Unique order_id: 96476
Duplicate order_id: 0

Missing target: 0


## Spara labelerat dataset

Det labelerade datasetet innehåller endast orders där target-variabeln
`late` kan bestämmas på ett tillförlitligt sätt.

Orders utan faktiskt leveransdatum har inte fått någon target och
utesluts därför från detta dataset.

Datasetet innehåller en rad per order och används som input till nästa
notebook för train-, validation- och testuppdelning.

In [46]:
# Spara det labelerade datasetet som artifact för nästa notebook
artifact_path = artifacts_dir / "labeled_ml_table.csv"

labeled_ml_table.to_csv(
    artifact_path,
    index=False
)

print("Saved:", artifact_path)
print("File exists:", artifact_path.exists())
print("Final shape:", labeled_ml_table.shape)

Saved: artifacts\labeled_ml_table.csv
File exists: True
Final shape: (96476, 21)


## SISTA RESULTAT

Target-variabeln `late` skapades genom att jämföra faktiskt
leveransdatum med uppskattat leveransdatum.

En order klassificeras som `late = 1` om den faktiska leveransen skedde
efter det uppskattade leveransdatumet. Om leveransen skedde på eller
före det uppskattade datumet får ordern `late = 0`.

Target-variabeln verifierades både genom manuella exempel och genom en
programmatisk kontroll. Inga felaktiga labels identifierades.

Av totalt 99 441 orders kunde 96 476 få en känd target. 2 965 orders
saknade faktiskt leveransdatum och kunde därför inte klassificeras på
ett tillförlitligt sätt.

Bland de labelerade orders är 88 649 (`91,89 %`) `late = 0` och 7 827
(`8,11 %`) `late = 1`. Datasetet har därför en tydlig class imbalance,
vilket behöver beaktas vid modellträning och utvärdering.

Det färdiga labelerade datasetet sparades som
`artifacts/labeled_ml_table.csv`.